In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import time
import optuna

from numba import njit

In [49]:
# ====================== JITTED BACKTEST CORE (stateful wealth & shares) ======================
@njit(fastmath=True, cache=True)
def simulate(closes: np.ndarray, pi_l_arr: np.ndarray, pi_u_arr: np.ndarray, c: float, initial_capital: float):
    n = len(closes)
    equity = np.empty(n, dtype=np.float64)
    trades = 0
    X = initial_capital
    n_shares = 0.0
    last_S = closes[0]
    equity[0] = X

    for k in range(1, n):
        S = closes[k]
        current_X = X + n_shares * (S - last_S)
        if current_X <= 0:
            current_X = 1e-8
        Y = n_shares * S
        pi_curr = Y / current_X

        pi_target = pi_curr
        delta = 0.0
        do_trade = False

        if pi_curr < pi_l_arr[k]:
            pi_target = pi_l_arr[k]
            delta = (pi_target * current_X - Y) / (1.0 + c * pi_target)
            do_trade = True
        elif pi_curr > pi_u_arr[k]:
            pi_target = pi_u_arr[k]
            delta = (pi_target * current_X - Y) / (1.0 - c * pi_target)
            do_trade = True

        if do_trade:
            abs_delta = abs(delta)
            new_Y = Y + delta
            new_X = current_X - c * abs_delta
            if new_X > 0 and S > 0:
                X = new_X
                n_shares = new_Y / S
                trades += 1
            else:
                n_shares = 0.0
                X = current_X
        else:
            X = current_X

        last_S = S
        equity[k] = X

    return equity, trades


def compute_metrics(equity: np.ndarray, initial_capital: float, n: int):
    final_pnl = equity[-1] - initial_capital
    if n < 2:
        return {'final_pnl': final_pnl, 'sharpe': 0.0, 'ann_sharpe': 0.0,
                'ann_turnover': 0.0, 'max_dd_pct': 0.0}

    returns = np.diff(equity) / equity[:-1]
    sharpe = 0.0
    ann_sharpe = 0.0
    if len(returns) > 30 and np.std(returns) > 1e-8:
        sharpe = np.mean(returns) / np.std(returns)
        annual_factor = 365.25 * 24
        ann_sharpe = sharpe * np.sqrt(annual_factor)

    cummax = np.maximum.accumulate(equity)
    dd = (equity - cummax) / cummax
    max_dd_pct = -dd.min() * 100 if dd.min() < 0 else 0.0

    return {
        'final_pnl': final_pnl,
        'sharpe': sharpe,
        'ann_sharpe': ann_sharpe,
        'max_dd_pct': max_dd_pct
    }


# ====================== BOUNDARY COMPUTATION (FIXED) ======================
def compute_boundaries(closes, M, gamma, shrink, c=0.001):
    n = len(closes)
    if n <= M + 1:
        return np.zeros(n, dtype=float), np.zeros(n, dtype=float)

    logrets = np.log(closes[1:] / closes[:-1])
    ret_s = pd.Series(logrets)

    # Force numpy for clean slicing
    roll_mean = ret_s.rolling(M, min_periods=M).mean().values
    roll_var = ret_s.rolling(M, min_periods=M).var(ddof=0).values

    mu_raw = roll_mean + 0.5 * roll_var
    mu_hat = (1.0 - shrink) * mu_raw + shrink * 0.0
    sigma_hat = np.sqrt(roll_var + 1e-12)

    pi_star = np.where(sigma_hat > 1e-8, mu_hat / (gamma * sigma_hat**2), 0.5)
    pi_star = np.clip(pi_star, 0.0, 1.0)

    d = (3 * pi_star**2 * (1 - pi_star)**2 / (4 * gamma)) ** (1.0/3) * (c ** (1.0/3))
    pi_l_roll = np.clip(pi_star - d, 0.0, 1.0)
    pi_u_roll = np.clip(pi_star + d, 0.0, 1.0)

    # First M bars: no-trade (all-cash)
    pi_l_arr = np.zeros(n, dtype=float)
    pi_u_arr = np.zeros(n, dtype=float)

    valid_start = M
    num_valid = n - valid_start
    if num_valid > 0:
        # pi_l_roll[M-1:] contains exactly the estimates for bars M..n-1
        pi_l_arr[valid_start:] = pi_l_roll[M-1 : M-1 + num_valid]
        pi_u_arr[valid_start:] = pi_u_roll[M-1 : M-1 + num_valid]

    return pi_l_arr, pi_u_arr

In [ ]:
csv_path = "data/BTCUSDT-1h-2023-01-01_2026-02-28.csv"   # ←←← UPDATE TO YOUR FILE

print("Loading CSV...")
df = pd.read_csv(csv_path)
df['datetime'] = pd.to_datetime(df["OpenTimestamp[ms]"], unit="ms")
df = df.sort_values("datetime").reset_index(drop=True)
n = len(df)
print(f"Data loaded: {n:,} minutes (~{n / (60 * 24):.1f} days)")

closes = df['Close'].values
initial_capital = 1_000_000.0
c = 0.001
M = 118
gamma = 0.86
shrink = 0.981

# ====================== DEFAULT BACKTEST ======================
print(f"\nRunning default (M={M}, γ={gamma}, shrink={shrink})...")
pi_l_def, pi_u_def = compute_boundaries(closes, M, gamma, shrink, c)
equity_before, trades_before = simulate(closes, pi_l_def, pi_u_def, 0, initial_capital)
equity_after, trades_after = simulate(closes, pi_l_def, pi_u_def, c, initial_capital)
metrics_before = compute_metrics(equity_before, trades_before, n)
metrics_after = compute_metrics(equity_after, initial_capital, n)


print("\n" + "="*80)
print("DEFAULT RESULTS (M=60, γ=3, shrink=0)")
print("="*80)
print(f"Final PnL BEFORE         : ${metrics_before['final_pnl']:,.2f}")
print(f"Final PnL AFTER          : ${metrics_after['final_pnl']:,.2f}")
print(f"Sharpe Ratio BEFORE      : {metrics_before['sharpe']:.4f}")
print(f"Sharpe Ratio AFTER       : {metrics_after['sharpe']:.4f}")
print(f"Annualized Sharpe BEFORE : {metrics_before['ann_sharpe']:.2f}")
print(f"Annualized Sharpe AFTER  : {metrics_after['ann_sharpe']:.2f}")
print(f"Max Drawdown BEFORE      : {metrics_before['max_dd_pct']:.2f}%")
print(f"Max Drawdown AFTER       : {metrics_after['max_dd_pct']:.2f}%")
print(f"Total trades BEFORE      : {trades_before:,}")
print(f"Total trades AFTER       : {trades_after:,}")
print("="*80)

# Save equity curve for Flask dashboard
equity_df = pd.DataFrame({'datetime': df['datetime'], 'pnl_before': equity_before - initial_capital, 'pnl_after': equity_after - initial_capital})
equity_df.to_csv('equity_curve.csv', index=False)
print("Saved 'equity_curve.csv' for the Flask dashboard.")

# Cumulative PnL plot (static)
plt.figure(figsize=(14, 7))
plt.plot(df['datetime'], equity_before - initial_capital, label='Before 0.1% cost', color='royalblue')
plt.plot(df['datetime'], equity_after - initial_capital, label='After 0.1% cost', color='crimson')
plt.title(f'Mean-Reversion Backtest (M={M}, γ={gamma}, shrink={shrink})', fontsize=16)
plt.xlabel('Time')
plt.ylabel('Cumulative PnL (USD)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# ====================== OPTUNA OBJECTIVE ======================
def objective(trial, closes, initial_capital, n, c=0.001):
    M = trial.suggest_int('M', 20, 200)
    gamma = trial.suggest_float('gamma', 0.5, 20.0)
    shrink = trial.suggest_float('shrink', 0.0, 1.0)

    pi_l_arr, pi_u_arr = compute_boundaries(closes, M, gamma, shrink, c)
    equity, trades = simulate(closes, pi_l_arr, pi_u_arr, c, initial_capital)
    metrics = compute_metrics(equity, initial_capital, n)
    return metrics['final_pnl']


# ====================== OPTUNA OPTIMIZATION ======================
n_trials = 1_000

print(f"\nStarting Optuna ({n_trials} trials) — tuning M, γ, James-Stein shrink...")
start_time = time.time()

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=67))
study.optimize(lambda trial: objective(trial, closes, initial_capital, n, c=0.001),
               n_trials=n_trials, show_progress_bar=True)

best = study.best_params
best_sharpe = study.best_value
print(f"\nOPTUNA BEST: M={best['M']}, γ={best['gamma']:.2f}, shrink={best['shrink']:.3f} → Sharpe = {best_sharpe:.4f}")
print(f"Finished in {time.time() - start_time:.1f}s")

# ====================== RUN BEST & SAVE ======================
print("\nRunning best parameters...")
pi_l_best, pi_u_best = compute_boundaries(closes, best['M'], best['gamma'], best['shrink'], c)
equity_best, trades_best = simulate(closes, pi_l_best, pi_u_best, c, initial_capital)
metrics_best = compute_metrics(equity_best, initial_capital, n)

equity_df = pd.DataFrame({
    'datetime': df['datetime'],
    'pnl': equity_best - initial_capital
})
equity_df.to_csv('equity_curve.csv', index=False)
print("✅ Saved 'equity_curve.csv' (BEST parameters)")

print("\n" + "="*80)
print(f"BEST RESULTS (M={best['M']}, γ={best['gamma']:.2f}, shrink={best['shrink']:.3f})")
print("="*80)
print(f"Final PnL         : ${metrics_best['final_pnl']:,.2f}")
print(f"Sharpe Ratio      : {metrics_best['sharpe']:.4f}")
print(f"Ann. Sharpe       : {metrics_best['ann_sharpe']:.2f}")
print(f"Max DD            : {metrics_best['max_dd_pct']:.2f}%")
print(f"Total trades      : {trades_best:,}")
print("="*80)

plt.figure(figsize=(14, 7))
plt.plot(df['datetime'], equity_best - initial_capital, color='royalblue', linewidth=1.5)
plt.title(f'BEST Merton Strategy (M={best["M"]}, γ={best["gamma"]:.2f}, shrink={best["shrink"]:.3f})', fontsize=16)
plt.xlabel('Time')
plt.ylabel('Cumulative PnL (USD)')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()